# Pipeline de Detección de Fraude — Demo

Este notebook ejecuta el pipeline completo (Bronze → Silver → Gold) sobre datos sintéticos de transacciones bancarias.

**Arquitectura**: AWS Glue (PySpark) + Delta Lake + Airflow

**Stack**: Medallion (Bronze/Silver/Gold)

## 1. Configuración e inicio de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import delta
import os, json, random, uuid
from datetime import datetime, timedelta

builder = SparkSession.builder \
    .appName("Fraude_Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = delta.configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark iniciado:", spark.version)

## 2. Generación de datos sintéticos

Generamos 50.000 transacciones sintéticas que replican la estructura de datos bancarios:
- `transaction_id`: identificador único
- `user_id`: UUID del cliente
- `card`: número de tarjeta (16 dígitos)
- `amount`: monto (con algunos valores ≤0 para probar el filtro de calidad)
- `transaction_date`: fecha de la transacción
- `timestamp`: datetime completo
- `merchant`: comercio
- `currency`: moneda

In [ ]:
ROWS = 50_000
USERS = [str(uuid.uuid4()) for _ in range(500)]
MERCHANTS = ["Amazon","Netflix","Spotify","Uber","Rappi","Falabella","MercadoLibre","Disney+","Apple","Google"]
CARDS = ["4532015112890367","4916123456789012","4556123456789013","4929123456789014","4539123456789015",
         "4485123456789016","4716123456789017","4024123456789018","4532123456789019","4913123456789020"]
CURRENCIES = ["USD","EUR","CLP","BRL"]

start = datetime(2026, 1, 1)
rows = []
for i in range(ROWS):
    amount = round(abs(random.gauss(150, 300)) + 0.01, 2)
    if i % 75 == 0:
        amount = random.choice([-amount, 0.0])
    tx_date = start + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))
    rows.append({
        "transaction_id": f"demo-{i:06d}",
        "user_id": random.choice(USERS),
        "card": random.choice(CARDS),
        "amount": amount,
        "transaction_date": tx_date.strftime("%Y-%m-%d"),
        "timestamp": tx_date.strftime("%Y-%m-%dT%H:%M:%S"),
        "merchant": random.choice(MERCHANTS),
        "currency": random.choice(CURRENCIES),
    })

df_raw = spark.createDataFrame(rows)
print(f"Filas generadas: {df_raw.count():,}")
df_raw.show(5, truncate=False)

## 3. Capa Bronze — Ingesta inmutable

Agregamos metadatos de linaje (`_ingestion_ts`, `_source_file`) y escribimos en formato Parquet.

**Decisión arquitectónica**: Bronze en Parquet simple (no Delta) porque es una capa de staging inmutable. Si se requiere replay, se relee desde la fuente original.

In [ ]:
BRONZE_PATH = "/tmp/demo/bronze/transactions/"

df_bronze = df_raw.withColumn("_ingestion_ts", F.current_timestamp()) \
                  .withColumn("_source_file", F.lit("demo_generated"))

df_bronze.write.mode("overwrite").parquet(BRONZE_PATH)

df_bronze_check = spark.read.parquet(BRONZE_PATH)
print(f"Bronze: {df_bronze_check.count():,} filas escritas en Parquet")
df_bronze_check.select("transaction_id", "amount", "_ingestion_ts", "_source_file").show(5, truncate=False)

## 4. Capa Silver — Calidad y enmascaramiento PII

Transformaciones aplicadas:
1. **Filtro de calidad**: solo transacciones con `amount > 0`
2. **Enmascaramiento PII**: el número de tarjeta se reemplaza por `4532-XXXX-XXXX-0367`, protegiendo datos sensibles
3. Escritura en **Delta Lake** con particionamiento por `transaction_date`

**Decisión arquitectónica**: Silver en Delta Lake para aprovechar ACID transactions, time travel, y compactación.

In [ ]:
SILVER_PATH = "/tmp/demo/silver/transactions/"

df_silver = df_bronze.filter(F.col("amount") > 0) \
    .withColumn("card_masked",
                F.concat(F.substring(F.col("card"), 1, 4),
                         F.lit("-XXXX-XXXX-"),
                         F.substring(F.col("card"), -4, 4))) \
    .drop("card")

df_silver.write.format("delta").mode("overwrite") \
    .partitionBy("transaction_date").save(SILVER_PATH)

df_silver_check = spark.read.format("delta").load(SILVER_PATH)
total = df_silver_check.count()
invalid = ROWS - total
print(f"Silver: {total:,} filas válidas (se eliminaron {invalid} filas con amount ≤ 0)")
df_silver_check.select("transaction_id", "amount", "card_masked").show(5, truncate=False)

### Verificación de enmascaramiento

Confirmamos que no existen números de tarjeta completos en la capa Silver.

In [ ]:
card_cols = [c for c in df_silver_check.columns if "card" in c.lower()]
print(f"Columnas relacionadas a tarjeta: {card_cols}")
assert "card" not in df_silver_check.columns, "ERROR: columna 'card' aún existe en Silver"
print("OK: columna 'card' eliminada correctamente.")

## 5. Capa Gold — Agregación de riesgo

Agregamos transacciones por `user_id` y `transaction_date` para generar un perfil de riesgo diario.

**Heurística**: un usuario se marca como `is_high_risk = True` cuando realiza más de 12 transacciones en un mismo día.

**Decisión arquitectónica**: Gold en Delta Lake para servir como fuente confiable a dashboards y modelos ML.

In [ ]:
GOLD_PATH = "/tmp/demo/gold/risk_profile/"

df_gold = df_silver_check.groupBy("user_id", "transaction_date") \
    .agg(F.count("*").alias("tx_count"),
         F.sum("amount").alias("daily_total")) \
    .withColumn("is_high_risk", F.col("tx_count") > 12)

df_gold.write.format("delta").mode("overwrite").save(GOLD_PATH)

df_gold_check = spark.read.format("delta").load(GOLD_PATH)
print(f"Gold: {df_gold_check.count():,} registros de perfil de riesgo")
df_gold_check.orderBy(F.col("tx_count").desc()).show(10, truncate=False)

## 6. Resultados

Resumen del pipeline completo.

In [ ]:
high_risk = df_gold_check.filter("is_high_risk = true").count()
total_users = df_gold_check.select("user_id").distinct().count()
print("=" * 50)
print("RESULTADOS DEL PIPELINE")
print("=" * 50)
print(f"  Bronze: {ROWS:,} filas raw")
print(f"  Silver: {total:,} filas válidas ({(total/ROWS)*100:.1f}% retención)")
print(f"  Gold:   {df_gold_check.count():,} perfiles de riesgo")
print(f"  Usuarios únicos: {total_users:,}")
print(f"  Usuarios high-risk (>12 tx/día): {high_risk:,}")
print("=" * 50)

print("\nTop 5 usuarios con más transacciones en un día:")
df_gold_check.orderBy(F.col("tx_count").desc()).select(
    "user_id", "transaction_date", "tx_count", "daily_total", "is_high_risk"
).show(5, truncate=False)

## 7. Limpieza

Eliminamos datos temporales del demo.

In [ ]:
import shutil
for p in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    shutil.rmtree(p, ignore_errors=True)
spark.stop()
print("Demo completado. Spark detenido.")